In [3]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw")
CLEAN_DIR = Path("/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(RAW_DIR.glob("*.csv"))
print(f"Found {len(files)} files")
files

Found 22 files


[PosixPath('/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw/BIL.csv'),
 PosixPath('/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw/ETHA.csv'),
 PosixPath('/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw/GLD.csv'),
 PosixPath('/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw/IBIT.csv'),
 PosixPath('/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw/IEI.csv'),
 PosixPath('/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw/INDA.csv'),
 PosixPath('/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw/QQQ.csv'),
 PosixPath('/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw/SHY.csv'),
 PosixPath('/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw/SLV.csv'),
 PosixPath('/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Raw/SPY.csv'),
 PosixPath('/Users/tradingw

In [4]:
def clean_etf_csv(file_path: Path) -> pd.DataFrame:
    symbol = file_path.stem.upper()

    df = pd.read_csv(file_path)

    # Standardize column names
    df = df.rename(columns={
        "Date": "date",
        "Price": "close",
        "CVol": "volume",
        "Open": "open",
        "Low": "low",
        "High": "high",
        "NAV": "nav",
        "Total Return (Gross)": "total_return_gross",
    })

    # Keep only needed columns if they exist
    keep_cols = [
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "nav",
        "total_return_gross",
    ]
    keep_cols = [col for col in keep_cols if col in df.columns]
    df = df[keep_cols].copy()

    # Add symbol
    df["symbol"] = symbol

    # Parse date
    df["date"] = pd.to_datetime(df["date"], format="%m/%d/%y", errors="coerce")

    # Convert numeric columns
    numeric_cols = [col for col in df.columns if col not in ["date", "symbol"]]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop bad rows
    df = df.dropna(subset=["date", "close"])

    # Sort oldest to newest
    df = df.sort_values("date").reset_index(drop=True)

    # Drop duplicate dates within symbol
    df = df.drop_duplicates(subset=["date"], keep="last")

    # Reorder columns
    ordered_cols = [
        "date",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "nav",
        "total_return_gross",
    ]
    ordered_cols = [col for col in ordered_cols if col in df.columns]
    df = df[ordered_cols]

    return df

In [5]:
all_dfs = []

for file_path in files:
    cleaned = clean_etf_csv(file_path)
    all_dfs.append(cleaned)
    print(f"{file_path.name}: {cleaned.shape[0]} rows")

master = pd.concat(all_dfs, ignore_index=True)
master = master.sort_values(["symbol", "date"]).reset_index(drop=True)

# Final duplicate check
master = master.drop_duplicates(subset=["symbol", "date"], keep="last")

print(master.shape)
master.head()

BIL.csv: 4721 rows
ETHA.csv: 406 rows
GLD.csv: 5356 rows
IBIT.csv: 538 rows
IEI.csv: 4818 rows
INDA.csv: 3539 rows
QQQ.csv: 6789 rows
SHY.csv: 5940 rows
SLV.csv: 4993 rows
SPY.csv: 8331 rows
TLT.csv: 5940 rows
TQQQ.csv: 4040 rows
UNG.csv: 3990 rows
USO.csv: 5007 rows
VEA.csv: 4682 rows
VFH.csv: 5559 rows
VHT.csv: 5559 rows
VIS.csv: 5392 rows
VIXY.csv: 1300 rows
VNQ.csv: 5392 rows
VPU.csv: 5559 rows
VWO.csv: 5280 rows
(103131, 9)


,date,symbol,open,high,low,close,volume,nav,total_return_gross
0,2007-06-01,BIL,91.62,91.62,91.62,91.62,1450.0,91.56,91.62
1,2007-06-04,BIL,91.64,91.64,91.64,91.64,1050.0,91.60,91.64
2,2007-06-05,BIL,91.66,91.68,91.64,91.64,3750.0,91.61,91.64
3,2007-06-06,BIL,91.64,91.68,91.64,91.68,9750.0,91.63,91.68
4,2007-06-07,BIL,91.64,91.66,91.64,91.66,12150.0,91.64,91.66


In [6]:
print("Symbols:", master["symbol"].nunique())
print("Date range:", master["date"].min(), "to", master["date"].max())
print("Duplicate symbol-date rows:", master.duplicated(subset=["symbol", "date"]).sum())
print("Missing closes:", master["close"].isna().sum())

Symbols: 22
Date range: 1993-02-01 00:00:00 to 2026-03-06 00:00:00
Duplicate symbol-date rows: 0
Missing closes: 0


In [7]:
output_csv = CLEAN_DIR / "etf_prices_master.csv"
output_parquet = CLEAN_DIR / "etf_prices_master.parquet"

master.to_csv(output_csv, index=False)
master.to_parquet(output_parquet, index=False)

print("Saved:")
print(output_csv)
print(output_parquet)

Saved:
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/etf_prices_master.csv
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/etf_prices_master.parquet


In [8]:
master = pd.concat(all_dfs, ignore_index=True)
master = master.sort_values(["symbol", "date"]).reset_index(drop=True)

# Final duplicate check
master = master.drop_duplicates(subset=["symbol", "date"], keep="last")

In [9]:
print(master.dtypes)

date                  datetime64[ns]
symbol                        object
open                         float64
high                         float64
low                          float64
close                        float64
volume                       float64
nav                          float64
total_return_gross           float64
dtype: object
